In [ ]:
import pandas as pd

df = pd.read_csv(
    "bank-full.csv",
    sep=";"
)

print(df.shape)
df.head()

In [ ]:
import matplotlib.pyplot as plt

df["y"].value_counts().plot(
    kind="bar"
)

plt.title("Subscription Distribution")
plt.xlabel("Subscription")
plt.ylabel("Customers")
plt.show()df["age"].plot(
    kind="hist",
    bins=30
)

plt.title("Age Distribution")
plt.xlabel("Age")
plt.ylabel("Customers")
plt.show()

In [ ]:
df["age"].plot(
    kind="hist",
    bins=30
)

plt.title("Age Distribution")
plt.xlabel("Age")
plt.ylabel("Customers")
plt.show()

In [ ]:
subscription_by_job = pd.crosstab(
    df["job"],
    df["y"],
    normalize="index"
) * 100

subscription_by_job

In [ ]:
df["balance"].describe()df["campaign"].describe()

In [ ]:
df["campaign"].describe()

In [ ]:
df_ml = df.drop(
    columns=["duration"]
).copy()

In [ ]:
df_ml["y"] = (
    df_ml["y"]
    .map({
        "no": 0,
        "yes": 1
    })
)

In [ ]:
df_ml["y"].value_counts()

In [ ]:
df_ml["total_contacts"] = (
    df_ml["campaign"] +
    df_ml["previous"]
)

In [ ]:
def age_group(age):

    if age < 30:
        return "18-29"

    elif age < 40:
        return "30-39"

    elif age < 50:
        return "40-49"

    elif age < 60:
        return "50-59"

    else:
        return "60+"


df_ml["age_group"] = df_ml["age"].apply(
    age_group
)

In [ ]:
df_ml.head()

In [ ]:
X = df_ml.drop(
    columns=["y"]
)

y = df_ml["y"]

In [ ]:
print("X:", X.shape)
print("y:", y.shape)

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

In [ ]:
categorical_features = X.select_dtypes(
    include=["object"]
).columns.tolist()

numerical_features = X.select_dtypes(
    exclude=["object"]
).columns.tolist()

print("Categorical:", categorical_features)
print("Numerical:", numerical_features)

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import (
    OneHotEncoder,
    StandardScaler
)

preprocessor = ColumnTransformer(
    transformers=[
        (
            "num",
            StandardScaler(),
            numerical_features
        ),
        (
            "cat",
            OneHotEncoder(
                handle_unknown="ignore"
            ),
            categorical_features
        )
    ]
)

In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression

logistic_model = Pipeline(
    steps=[
        (
            "preprocessor",
            preprocessor
        ),
        (
            "classifier",
            LogisticRegression(
                max_iter=2000,
                class_weight="balanced",
                random_state=42
            )
        )
    ]
)

logistic_model.fit(
    X_train,
    y_train
)

In [ ]:
from sklearn.ensemble import RandomForestClassifier

random_forest_model = Pipeline(
    steps=[
        (
            "preprocessor",
            preprocessor
        ),
        (
            "classifier",
            RandomForestClassifier(
                n_estimators=300,
                max_depth=12,
                min_samples_leaf=3,
                class_weight="balanced_subsample",
                random_state=42,
                n_jobs=-1
            )
        )
    ]
)

random_forest_model.fit(
    X_train,
    y_train
)

In [ ]:
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    average_precision_score
)

In [ ]:
def evaluate(
    name,
    model,
    X_test,
    y_test
):

    pred = model.predict(X_test)

    probability = model.predict_proba(
        X_test
    )[:, 1]

    return {
        "Model": name,
        "Accuracy": accuracy_score(
            y_test,
            pred
        ),
        "Precision": precision_score(
            y_test,
            pred
        ),
        "Recall": recall_score(
            y_test,
            pred
        ),
        "F1": f1_score(
            y_test,
            pred
        ),
        "ROC_AUC": roc_auc_score(
            y_test,
            probability
        ),
        "PR_AUC": average_precision_score(
            y_test,
            probability
        )
    }

In [ ]:
results = pd.DataFrame([
    evaluate(
        "Logistic Regression",
        logistic_model,
        X_test,
        y_test
    ),

    evaluate(
        "Random Forest",
        random_forest_model,
        X_test,
        y_test
    )
])

results

In [ ]:
best_row = results.loc[
    results["PR_AUC"].idxmax()
]

best_row

In [ ]:
if best_row["Model"] == "Random Forest":

    best_model = random_forest_model

else:

    best_model = logistic_modelprint(
    "Selected model:",
    best_row["Model"]
)

In [ ]:
print(
    "Selected model:",
    best_row["Model"]
)

In [ ]:
from sklearn.metrics import confusion_matrix

predictions = best_model.predict(
    X_test
)

cm = confusion_matrix(
    y_test,
    predictions
)

print(cm)

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(6, 5))

plt.imshow(cm)

plt.title("Confusion Matrix")
plt.xlabel("Predicted")
plt.ylabel("Actual")

plt.xticks(
    [0, 1],
    ["No", "Yes"]
)

plt.yticks(
    [0, 1],
    ["No", "Yes"]
)

plt.colorbar()

plt.show()

In [ ]:
sample_customer = pd.DataFrame([
    {
        "age": 35,
        "job": "management",
        "marital": "single",
        "education": "tertiary",
        "default": "no",
        "balance": 2500,
        "housing": "yes",
        "loan": "no",
        "contact": "cellular",
        "day": 15,
        "month": "may",
        "campaign": 1,
        "pdays": -1,
        "previous": 0,
        "poutcome": "unknown",
        "total_contacts": 1,
        "age_group": "30-39"
    }
])

In [ ]:
prediction = best_model.predict(
    sample_customer
)[0]

probability = best_model.predict_proba(
    sample_customer
)[0, 1]

print("Prediction:", prediction)
print(
    "Probability:",
    f"{probability:.2%}"
)

In [ ]:
from snowflake.ml.registry import Registry

session = session

registry = Registry(
    session=session,
    database_name="BANK_ML_DB",
    schema_name="ML"
)

In [ ]:
from snowflake.snowpark.context import get_active_session
from snowflake.ml.registry import Registry

# Get the active Snowflake session
session = get_active_session()

# Create the Model Registry
registry = Registry(
    session=session,
    database_name="BANK_ML_DB",
    schema_name="ML"
)

print("Model Registry connected successfully!")

In [ ]:
model_ref = registry.log_model(
    model=best_model,
    model_name="BANK_RESPONSE_MODEL",
    version_name="V1",
    sample_input_data=X_test.head(5)
)

print("Model registered successfully!")

In [ ]:
registry.get_model(
    "BANK_RESPONSE_MODEL"
).show_versions()

In [ ]:
import snowflake.ml
print("snowflake-ml-python is available")

In [ ]:
%%sql -r dataframe_1
SELECT CURRENT_WAREHOUSE();

In [ ]:
# Get V1 from the Model Registry
registered_model = (
    registry
    .get_model("BANK_RESPONSE_MODEL")
    .version("V1")
)

print("Registered model loaded successfully!")

In [ ]:
prediction_result = registered_model.run(
    sample_customer,
    function_name="predict"
)

prediction_result

In [ ]:
%%sql -r dataframe_2
USE WAREHOUSE BANK_ML_WH;

In [ ]:
%%sql -r dataframe_3
SELECT CURRENT_WAREHOUSE();

In [ ]:
model_ref_v2 = registry.log_model(
    model=best_model,
    model_name="BANK_RESPONSE_MODEL",
    version_name="V2",
    sample_input_data=X_test.head(5),
    target_platforms=["WAREHOUSE"],
    options={
        "method_options": {
            "predict": {
                "case_sensitive": True
            },
            "predict_proba": {
                "case_sensitive": True
            }
        }
    }
)

print("V2 registered for Warehouse inference!")

In [ ]:
from snowflake.snowpark.context import get_active_session

# Get the active notebook session
session = get_active_session()

# Explicitly set the session context
session.use_database("BANK_ML_DB")
session.use_schema("ML")
session.use_warehouse("BANK_ML_WH")

print("Database :", session.get_current_database())
print("Schema   :", session.get_current_schema())
print("Warehouse:", session.get_current_warehouse())

In [ ]:
from snowflake.ml.registry import Registry

registry = Registry(
    session=session,
    database_name="BANK_ML_DB",
    schema_name="ML"
)

print("Registry connected successfully!")

In [ ]:
versions = registry.get_model(
    "BANK_RESPONSE_MODEL"
).show_versions()

versions

In [ ]:
model_ref_v2 = registry.log_model(
    model=best_model,
    model_name="BANK_RESPONSE_MODEL",
    version_name="V2",
    sample_input_data=X_test.head(5),
    target_platforms=["WAREHOUSE"],
    options={
        "method_options": {
            "predict": {
                "case_sensitive": True
            },
            "predict_proba": {
                "case_sensitive": True
            }
        }
    }
)

print("V2 registered successfully for Warehouse inference!")

In [ ]:
target_platforms=["WAREHOUSE"]

In [ ]:
registry.get_model(
    "BANK_RESPONSE_MODEL"
).show_versions()

In [ ]:
registered_model_v2 = (
    registry
    .get_model("BANK_RESPONSE_MODEL")
    .version("V2")
)

print("V2 loaded successfully!")

In [ ]:
prediction_result = registered_model_v2.run(
    sample_customer,
    function_name="predict"
)

prediction_result

In [ ]:
"case_sensitive": True

In [ ]:
registered_model_v2.show_functions()

In [ ]:
prediction_result = registered_model_v2.run(
    sample_customer,
    function_name='"predict"'
)

prediction_result

In [ ]:
prediction_result = registered_model_v2.run(
    sample_customer,
    function_name='"predict"'
)

probability_result = registered_model_v2.run(
    sample_customer,
    function_name='"predict_proba"'
)

predicted_class = int(prediction_result.iloc[0, 0])
subscribe_probability = float(probability_result.iloc[0, 1])

print(
    "Prediction:",
    "SUBSCRIBE" if predicted_class == 1
    else "NOT SUBSCRIBE"
)

print(
    f"Subscription Probability: "
    f"{subscribe_probability:.2%}"
)

In [ ]:
model_ref_v2.set_metric(
    "accuracy",
    float(best_row["Accuracy"])
)

model_ref_v2.set_metric(
    "precision",
    float(best_row["Precision"])
)

model_ref_v2.set_metric(
    "recall",
    float(best_row["Recall"])
)

model_ref_v2.set_metric(
    "f1",
    float(best_row["F1"])
)

model_ref_v2.set_metric(
    "roc_auc",
    float(best_row["ROC_AUC"])
)

model_ref_v2.set_metric(
    "pr_auc",
    float(best_row["PR_AUC"])
)

print("Metrics saved to V2")

In [ ]:
registered_model_v2.show_metrics()

In [ ]:
registry.get_model(
    "BANK_RESPONSE_MODEL"
).show_versions()

In [ ]:
%%sql -r dataframe_4
ALTER MODEL BANK_ML_DB.ML.BANK_RESPONSE_MODEL
SET DEFAULT_VERSION = 'V2';